In [1]:
from pathlib import Path
import pandas as pd


In [13]:
ROOT_DIR = Path(".").resolve().parent
model_dir = ROOT_DIR / "results/cgcnn_models/TSD_SSD_WS24_water_WS24_water4_WS24_acid_WS24_base_WS24_boiling_seed42_att_cgcnn/version_43"
model_dir

PosixPath('/home/zhangsd/repos/MOFSNN/results/cgcnn_models/TSD_SSD_WS24_water_WS24_water4_WS24_acid_WS24_base_WS24_boiling_seed42_att_cgcnn/version_43')

In [92]:
df_water = pd.read_csv(model_dir / "external_test_results_WS24_water.csv")
df_water4 = pd.read_csv(model_dir / "external_test_results_WS24_water4.csv")

# df_water = pd.read_csv(ROOT_DIR / "results/ml_models/WS24/RAC_and_zeo_features_with_id_prop/water_label/external_test_predicted_RandomForestClassifier.csv")
# df_water4 = pd.read_csv(ROOT_DIR / "results/ml_models/WS24/RAC_and_zeo_features_with_id_prop/water4_label/external_test_predicted_RandomForestClassifier.csv")

In [93]:
df_water.rename(columns={"GroundTruth": "GroundTruthWater", "Predicted": "PredictedWater", "Uncertainty": "UncertaintyWater"}, inplace=True)
df_water.head(2)


,CifId,GroundTruthWater,PredictedWater,Prob,UncertaintyWater
0,val_1,1,1,0.999975,0.038991
1,val_2,1,1,0.999986,0.043611


In [94]:
df_water_all = df_water4.merge(
    df_water[["CifId", "GroundTruthWater", "PredictedWater", "UncertaintyWater"]],
    on="CifId",
    how="left",
)
df_water_all.head(2)

,CifId,GroundTruth,Predicted,Prob,Uncertainty,GroundTruthWater,PredictedWater,UncertaintyWater
0,val_1,2,2,"[5.883174253540346e-06, 0.00011709304817486554...",0.107084,1,1,0.038991
1,val_2,2,2,"[2.520262114558136e-06, 8.534711378160864e-05,...",0.131245,1,1,0.043611


In [95]:
df_water_all["Water4to2"] = df_water_all["Predicted"].apply(lambda x: 1 if x > 1 else 0)
df_water_all["Consistent"] = df_water_all["PredictedWater"] == df_water_all["Water4to2"]

In [96]:
## Calculate total accuracy of water4 predictions
total_accuracy = (df_water_all["GroundTruth"] == df_water_all["Predicted"]).sum() / len(df_water_all)
print(f"Total accuracy: {total_accuracy:.2%} in {len(df_water_all)} predictions")
## calculate accuracy for consistent predictions
consistent_accuracy = (df_water_all["Consistent"] & (df_water_all["GroundTruth"] == df_water_all["Predicted"])).sum() / df_water_all["Consistent"].sum()
print(f"Consistent accuracy: {consistent_accuracy:.2%} in {df_water_all['Consistent'].sum()} consistent predictions")

## Calculate accuracy for inconsistent predictions
inconsistent_accuracy = ((~df_water_all["Consistent"]) & (df_water_all["GroundTruth"] == df_water_all["Predicted"])).sum() / (~df_water_all["Consistent"]).sum()
print(f"Inconsistent accuracy: {inconsistent_accuracy:.2%} in {(~df_water_all['Consistent']).sum()} inconsistent predictions")
## print inconsistent predictions
inconsistent_predictions = df_water_all[~df_water_all["Consistent"]]
print(f'Inconsistent predictions: \n{inconsistent_predictions[["CifId", "GroundTruth", "Predicted", "PredictedWater"]]}')

## Calculate accuracy for low uncertainty predictions
uncertainty_cutoff = 0.92 ## This is a threshold that can retain 80% of test set samples in water4 task
low_uncertainty_accuracy = (df_water_all[df_water_all["Uncertainty"] < uncertainty_cutoff]["GroundTruth"] == df_water_all[df_water_all["Uncertainty"] < uncertainty_cutoff]["Predicted"]).sum() / len(df_water_all[df_water_all["Uncertainty"] < uncertainty_cutoff])
print(f"Low uncertainty accuracy: {low_uncertainty_accuracy:.2%} in {len(df_water_all[df_water_all['Uncertainty'] < uncertainty_cutoff])} low uncertainty predictions (cutoff: {uncertainty_cutoff})")
## Calculate accuracy for high uncertainty predictions
high_uncertainty_accuracy = (df_water_all[df_water_all["Uncertainty"] >= uncertainty_cutoff]["GroundTruth"] == df_water_all[df_water_all["Uncertainty"] >= uncertainty_cutoff]["Predicted"]).sum() / len(df_water_all[df_water_all["Uncertainty"] >= uncertainty_cutoff])
print(f"High uncertainty accuracy: {high_uncertainty_accuracy:.2%} in {len(df_water_all[df_water_all['Uncertainty'] >= uncertainty_cutoff])} high uncertainty predictions (cutoff: {uncertainty_cutoff})")
## Calculate accuracy for low uncertainty consistent predictions
low_uncertainty_consistent_accuracy = (df_water_all[(df_water_all["Uncertainty"] < uncertainty_cutoff) & df_water_all["Consistent"]]["GroundTruth"] == df_water_all[(df_water_all["Uncertainty"] < uncertainty_cutoff) & df_water_all["Consistent"]]["Predicted"]).sum() / len(df_water_all[(df_water_all["Uncertainty"] < uncertainty_cutoff) & df_water_all["Consistent"]])
print(f"Low uncertainty consistent accuracy: {low_uncertainty_consistent_accuracy:.2%} in {len(df_water_all[(df_water_all['Uncertainty'] < uncertainty_cutoff) & df_water_all['Consistent']])} low uncertainty consistent predictions (cutoff: {uncertainty_cutoff})")

Total accuracy: 39.13% in 46 predictions
Consistent accuracy: 42.86% in 42 consistent predictions
Inconsistent accuracy: 0.00% in 4 inconsistent predictions
Inconsistent predictions: 
     CifId  GroundTruth  Predicted  PredictedWater
3    val_4            3          1               1
19  val_23            3          1               1
29  val_33            3          1               1
43  val_47            1          0               1
Low uncertainty accuracy: 44.12% in 34 low uncertainty predictions (cutoff: 0.92)
High uncertainty accuracy: 25.00% in 12 high uncertainty predictions (cutoff: 0.92)
Low uncertainty consistent accuracy: 45.45% in 33 low uncertainty consistent predictions (cutoff: 0.92)


In [98]:
## Make table to present results above
results_table = pd.DataFrame({
    "Metric": [
        "Total Accuracy",
        "Accuracy of Consistent Predictions",
        "Accuracy of Inconsistent Predictions",
        f"Accuracy of Low Uncertainty (<{uncertainty_cutoff}) Predictions",
        f"Accuracy of High Uncertainty (≧{uncertainty_cutoff}) Predictions",
        f"Accuracy of Low Uncertainty Consistent Predictions"
    ],
    "Value": [
        f"{total_accuracy:.2%}",
        f"{consistent_accuracy:.2%}",
        f"{inconsistent_accuracy:.2%}",
        f"{low_uncertainty_accuracy:.2%}",
        f"{high_uncertainty_accuracy:.2%}",
        f"{low_uncertainty_consistent_accuracy:.2%}"
    ],
    "Samples": [
        len(df_water_all),
        df_water_all["Consistent"].sum(),
        (~df_water_all["Consistent"]).sum(),
        len(df_water_all[df_water_all["Uncertainty"] < uncertainty_cutoff]),
        len(df_water_all[df_water_all["Uncertainty"] >= uncertainty_cutoff]),
        len(df_water_all[(df_water_all["Uncertainty"] < uncertainty_cutoff) & df_water_all["Consistent"]])
    ]
})
print("\nResults Table:")
results_table
# Save the results table to a CSV file


Results Table:


,Metric,Value,Samples
0,Total Accuracy,39.13%,46
1,Accuracy of Consistent Predictions,42.86%,42
2,Accuracy of Inconsistent Predictions,0.00%,4
3,Accuracy of Low Uncertainty (<0.92) Predictions,44.12%,34
4,Accuracy of High Uncertainty (≧0.92) Predictions,25.00%,12
5,Accuracy of Low Uncertainty Consistent Predict...,45.45%,33
